# CAGA v30.2 — MMAR, KL loss, 5 modes (NVIDIA A100 80GB)

**What this notebook does**: runs steering on **MMAR** with loss = **KL**, K = **10000**, 5 modes (`final / audio / text / audio_text / per_token`), alpha swept over `-1 ~ +1`, then builds tables and saves Excel to `$CAGA_RUNS`.

**GPU requirement**: Qwen2.5-Omni-7B in bf16 needs about 16 GB. **The free T4 (16 GB) runs out of memory**; use **L4 (24 GB) or A100**. If it still OOMs, set `partb.FREE_CACHE=True` and `partb.BATCH_ALPHAS=False`.

**Preparation**: upload `caga_package.zip` to `$CAGA_BASE`. Cell 1 finds and extracts it automatically.

In [ ]:
# 1) Set paths and locate the package
import os, sys, glob, zipfile
os.environ.setdefault("CAGA_BASE", os.path.expanduser("~/caga"))
BASE = os.environ["CAGA_BASE"]
ROOT = os.path.join(BASE, "caga_v30_2")
if not os.path.isdir(os.path.join(ROOT, "caga")):
    zips = glob.glob(os.path.join(BASE, "caga_package.zip"))
    assert zips, "caga_v30_2/ or caga_package.zip not found"
    with zipfile.ZipFile(zips[0]) as z: z.extractall(BASE)
sys.path.insert(0, ROOT)
print("package root:", ROOT)

package root: $CAGA_BASE/caga_v30_2


In [ ]:
# 2) Install dependencies
!pip install -q "transformers>=4.52" accelerate qwen-omni-utils torchcodec huggingface_hub hf_transfer datasets soundfile librosa requests tqdm
!apt-get -qq install -y ffmpeg >/dev/null 2>&1
print("deps ready")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 69.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.8/35.8 MB 74.5 MB/s eta 0:00:00
deps ready


### 3) Download MMAR (about 2.8 GB, no GPU needed)

In [ ]:
from caga.data import build, data_summary
build('mmar')
data_summary(['mmar'])


########## building mmar ##########


/usr/local/lib/python3.13/dist-packages/huggingface_hub/constants.py:299: FutureWarning: The `HF_HUB_ENABLE_HF_TRANSFER` environment variable is deprecated as 'hf_transfer' is not used anymore. Please use `HF_XET_HIGH_PERFORMANCE` instead to enable high performance transfer with Xet. Visit https://huggingface.co/docs/huggingface_hub/package_reference/environment_variables#hfxethighperformance for more details.
  warnings.warn(


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

  [extracted] mmar-audio.tar.gz -> $CAGA_BASE/datasets/mmar/raw
[mmar] wrote 1000 rows -> $CAGA_BASE/datasets/mmar/manifest.jsonl
[mmar] 1000 items | missing audio: 0
   e.g. {'id': 'mmar/qx8hrhBZJ98_00-01-32_00-02-02', 'domain': 'music', 'choices': ['Mountain Canyon', 'Dense Forest', 'Open Grassland', 'City Streets'], 'gold_idx': 2}
[mmar] matched 1000/1000 (official GitHub meta, exact gold)
dataset                  items  audio_ok   status
------------------------------------------------------
mmar                      1000      1000   OK

schema: id · dataset · audio_path · question · choices · gold_idx · domain


### 4) Run steering (KL · K=10000 · 5 modes · full alpha sweep)
1000 items take about 1.5-2.5 hours. Resumable.

### 4a) Selection criterion SELECT_SCORE (one at a time, change it here)
- `contrast` = |salA-salB| (original) · `clean_abs` = |salA| · `clean_signed` = salA
- `noise_abs` = |salB| · `noise_signed` = salB · `aad` = actA-actB (no gradient, advanced baseline)

Each SELECT_SCORE writes to a **separate directory** (contrast keeps the original name, the others get a `_{name}` suffix), so they never overwrite each other.
To compare several: change this cell -> rerun 4b/5 -> next one.

In [ ]:
from caga import partb
partb.SELECT_SCORE = 'clean_signed'    # <- change this: contrast / clean_abs / clean_signed / noise_abs / noise_signed / aad
print("SELECT_SCORE =", partb.SELECT_SCORE, "| uses gradient:", partb.SELECT_SCORE!='aad')

config ready | contrast = gaussian_pure | alphas = [-1.0, -0.3, 0.0, 0.3, 1.0] | RUNS_BASE = $CAGA_BASE/caga_v30_2_runs
forward helpers ready (gaussian 'pure' contrast)
v30 methods ready: compute_scope_attribution (gradxact) / build_specs (modes) / PerItemRunner
SELECT_SCORE = clean_signed | uses gradient: True


In [ ]:
from caga import partb
partb.load_model()
partb.run_v30_2(['mmar'])
partb.eval_v30_2(['mmar'])

[threads] intra-op=8 | affinity cores=12
[Load] Qwen/Qwen2.5-Omni-7B


chat_template.json:   0%|          | 0.00/1.31k [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/667 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/13.2k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/6.47k [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/1346 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/74.0 [00:00<?, ?B/s]

[Load] done

=== mmar: 1000 items | modes=['final', 'audio', 'text', 'audio_text', 'per_token'] | K=[10000] | 5 configs x 5 alphas | 2 fwd-bwd/item (gradxact) ===
  [sanity] id=mmar/qx8hrhBZJ98_00-01-32_00-02-02 grad-L2 audio/pos=0.01522 text/pos=0.37150 final=1.123 T=810
    [mmar] 10 items (101s)
    [mmar] 20 items (176s)
    [mmar] 30 items (258s)
    [mmar] 40 items (335s)
    [mmar] 50 items (398s)
    [mmar] 60 items (463s)
    [mmar] 70 items (529s)
    [mmar] 80 items (607s)
    [mmar] 90 items (673s)
    [mmar] 100 items (753s)
    [mmar] 110 items (820s)
    [mmar] 120 items (886s)
    [mmar] 130 items (959s)
    [mmar] 140 items (1036s)
    [mmar] 150 items (1113s)
    [mmar] 160 items (1170s)
    [mmar] 170 items (1241s)
    [mmar] 180 items (1303s)
    [mmar] 190 items (1372s)
    [mmar] 200 items (1452s)
    [mmar] 210 items (1519s)
    [mmar] 220 items (1592s)
    [mmar] 230 items (1662s)
    [mmar] 240 items (1738s)
    [mmar] 250 items (1825s)
    [mmar] 260 items (19

### 5) Build tables + save Excel

In [ ]:
import os, json
import pandas as pd
from caga import partb
from caga.adapter import alpha_tag
DS='mmar'; K=partb.V30_K_LIST[0]
base_dir = os.path.join(partb.RUNS_BASE, "caga_eval_v30_2", DS)
ALPHAS = partb.V30_ALPHAS; MODES = partb.V30_MODES
def acc_of(mode, a):
    p = os.path.join(partb._v30_dir(base_dir, mode, K), f"alpha_{alpha_tag(a)}.jsonl")
    if not os.path.exists(p): return (float('nan'), 0)
    n=c=0
    for line in open(p):
        line=line.strip()
        if not line: continue
        r=json.loads(line); n+=1; c+=int(bool(r.get('correct')))
    return (c/n if n else float('nan'), n)
rows=[]
for mode in MODES:
    rec={'mode':mode}; accs={}
    for a in ALPHAS:
        acc,n=acc_of(mode,a); accs[a]=acc
        rec[f"a={a:+.1f}"]=round(acc,4) if acc==acc else None
    base=accs.get(0.0,float('nan'))
    nz={a:accs[a] for a in ALPHAS if a!=0.0 and accs[a]==accs[a]}
    if nz:
        ba=max(nz,key=nz.get); rec['best_acc']=round(nz[ba],4); rec['best_alpha']=f"{ba:+.1f}"
        rec['delta_vs0']=round(nz[ba]-base,4) if base==base else None
    rec['n']=acc_of(mode,0.0)[1]; rows.append(rec)
df=pd.DataFrame(rows)
out=os.path.join(partb.RUNS_BASE, f"{DS}_KL_K{K}_results.xlsx"); df.to_excel(out,index=False)
print("saved:", out); df

saved: $CAGA_BASE/caga_v30_2_runs/mmar_KL_K10000_results.xlsx


         mode  a=-1.0  a=-0.3  a=+0.0  a=+0.3  a=+1.0  best_acc best_alpha  \
0       final   0.627   0.639   0.629   0.626   0.632     0.639       -0.3   
1       audio   0.335   0.393   0.629   0.599   0.582     0.599       +0.3   
2        text   0.380   0.538   0.629   0.623   0.601     0.623       +0.3   
3  audio_text   0.291   0.354   0.629   0.592   0.578     0.592       +0.3   
4   per_token   0.263   0.278   0.629   0.579   0.561     0.579       +0.3   

   delta_vs0     n  
0      0.010  1000  
1     -0.030  1000  
2     -0.006  1000  
3     -0.037  1000  
4     -0.050  1000  

In [ ]:
# Scan all finished SELECT_SCOREs into one summary table
import os, json, glob
import pandas as pd
from caga import partb
from caga.adapter import alpha_tag

DS='mmar'; K=partb.V30_K_LIST[0]; LOSS='KL'
base_dir = os.path.join(partb.RUNS_BASE, "caga_eval_v30_2", DS)
ALPHAS = partb.V30_ALPHAS

# Recover (select_score, mode) from the directory name:  gradxact[_{select}]_{mode}_none_K{K}[_pt]
def parse_dir(name):
    m = name[len("gradxact"):]                     # _{select}_{mode}_none_K10000
    assert m.startswith("_") or m.startswith("_"), name
    core = m.rsplit(f"_none_K{K}", 1)[0].lstrip("_")   # {select}_{mode} or {mode}
    KNOWN_MODES = ['final','audio','text','audio_text','per_token']
    for mode in sorted(KNOWN_MODES, key=len, reverse=True):
        if core == mode:            return ('contrast', mode)      # no select prefix = contrast
        if core.endswith("_"+mode): return (core[:-(len(mode)+1)], mode)
    return (core, '?')

def acc_of(d, a):
    p = os.path.join(d, f"alpha_{alpha_tag(a)}.jsonl")
    if not os.path.exists(p): return (float('nan'), 0)
    n=c=0
    for line in open(p):
        line=line.strip()
        if not line: continue
        r=json.loads(line); n+=1; c+=int(bool(r.get('correct')))
    return (c/n if n else float('nan'), n)

rows=[]
for d in sorted(glob.glob(os.path.join(base_dir, f"gradxact*_none_K{K}*"))):
    sel, mode = parse_dir(os.path.basename(d))
    rec={'dataset':DS,'loss':LOSS,'select_score':sel,'mode':mode}
    accs={}
    for a in ALPHAS:
        acc,n=acc_of(d,a); accs[a]=acc
        rec[f"a={a:+.1f}"]=round(acc,4) if acc==acc else None
    base=accs.get(0.0,float('nan'))
    nz={a:accs[a] for a in ALPHAS if a!=0.0 and accs[a]==accs[a]}
    if nz:
        ba=max(nz,key=nz.get); rec['best_acc']=round(nz[ba],4)
        rec['best_alpha']=f"{ba:+.1f}"; rec['delta_vs0']=round(nz[ba]-base,4) if base==base else None
    rec['n']=acc_of(d,0.0)[1]; rows.append(rec)

df=pd.DataFrame(rows).sort_values(['select_score','mode']).reset_index(drop=True)
out=os.path.join(partb.RUNS_BASE, f"{DS}_{LOSS}_ALL_selectscores_K{K}.xlsx")
df.to_excel(out, index=False)
print(f"{df['select_score'].nunique()} select_score(s) x {df['mode'].nunique()} modes | saved: {out}")
df

1 select_score(s) x 5 modes | saved: $CAGA_BASE/caga_v30_2_runs/mmar_KL_ALL_selectscores_K10000.xlsx


  dataset loss  select_score        mode  a=-1.0  a=-0.3  a=+0.0  a=+0.3  \
0    mmar   KL  clean_signed       audio   0.335   0.393   0.629   0.599   
1    mmar   KL  clean_signed  audio_text   0.291   0.354   0.629   0.592   
2    mmar   KL  clean_signed       final   0.627   0.639   0.629   0.626   
3    mmar   KL  clean_signed   per_token   0.263   0.278   0.629   0.579   
4    mmar   KL  clean_signed        text   0.380   0.538   0.629   0.623   

   a=+1.0  best_acc best_alpha  delta_vs0     n  
0   0.582     0.599       +0.3     -0.030  1000  
1   0.578     0.592       +0.3     -0.037  1000  
2   0.632     0.639       -0.3      0.010  1000  
3   0.561     0.579       +0.3     -0.050  1000  
4   0.601     0.623       +0.3     -0.006  1000  

To run another dataset: replace `'mmar'` in cells 3-5.

In [ ]:
# Run the other SELECT_SCOREs in one go (sequential, resumable, finished ones are skipped)
from caga import partb

TO_RUN = ['aad', 'clean_abs', 'noise_signed']  # list everything to run here
DS = 'mmar'

partb.load_model()   # returns immediately if already loaded, no reload
for sel in TO_RUN:
    partb.SELECT_SCORE = sel
    print(f"\n########## SELECT_SCORE = {sel} (uses gradient: {sel!='aad'}) ##########")
    partb.run_v30_2([DS])       # writes to its own directory; finished (mode, K, alpha, item) are skipped
    partb.eval_v30_2([DS])      # print the text table for this select score
print("\n===== all SELECT_SCORE runs finished =====")


########## SELECT_SCORE = aad (uses gradient: False) ##########

=== mmar: 1000 items | modes=['final', 'audio', 'text', 'audio_text', 'per_token'] | K=[10000] | 5 configs x 5 alphas | 2 fwd-bwd/item (gradxact) ===
  [sanity] id=mmar/qx8hrhBZJ98_00-01-32_00-02-02 grad-L2 audio/pos=0.00000 text/pos=0.00000 final=0.000 T=810
    [mmar] 10 items (50s)
    [mmar] 20 items (96s)
    [mmar] 30 items (150s)
    [mmar] 40 items (200s)
    [mmar] 50 items (242s)
    [mmar] 60 items (284s)
    [mmar] 70 items (328s)
    [mmar] 80 items (378s)
    [mmar] 90 items (421s)
    [mmar] 100 items (475s)
    [mmar] 110 items (520s)
    [mmar] 120 items (565s)
    [mmar] 130 items (612s)
    [mmar] 140 items (663s)
    [mmar] 150 items (715s)
    [mmar] 160 items (754s)
    [mmar] 170 items (803s)
    [mmar] 180 items (845s)
    [mmar] 190 items (891s)
    [mmar] 200 items (943s)
    [mmar] 210 items (989s)
    [mmar] 220 items (1038s)
    [mmar] 230 items (1084s)
    [mmar] 240 items (1134s)
    [mmar]

In [ ]:
# Scan all finished SELECT_SCOREs into one summary table
import os, json, glob
import pandas as pd
from caga import partb
from caga.adapter import alpha_tag

DS='mmar'; K=partb.V30_K_LIST[0]; LOSS='KL'
base_dir = os.path.join(partb.RUNS_BASE, "caga_eval_v30_2", DS)
ALPHAS = partb.V30_ALPHAS

# Recover (select_score, mode) from the directory name:  gradxact[_{select}]_{mode}_none_K{K}[_pt]
def parse_dir(name):
    m = name[len("gradxact"):]                     # _{select}_{mode}_none_K10000
    assert m.startswith("_") or m.startswith("_"), name
    core = m.rsplit(f"_none_K{K}", 1)[0].lstrip("_")   # {select}_{mode} or {mode}
    KNOWN_MODES = ['final','audio','text','audio_text','per_token']
    for mode in sorted(KNOWN_MODES, key=len, reverse=True):
        if core == mode:            return ('contrast', mode)      # no select prefix = contrast
        if core.endswith("_"+mode): return (core[:-(len(mode)+1)], mode)
    return (core, '?')

def acc_of(d, a):
    p = os.path.join(d, f"alpha_{alpha_tag(a)}.jsonl")
    if not os.path.exists(p): return (float('nan'), 0)
    n=c=0
    for line in open(p):
        line=line.strip()
        if not line: continue
        r=json.loads(line); n+=1; c+=int(bool(r.get('correct')))
    return (c/n if n else float('nan'), n)

rows=[]
for d in sorted(glob.glob(os.path.join(base_dir, f"gradxact*_none_K{K}*"))):
    sel, mode = parse_dir(os.path.basename(d))
    rec={'dataset':DS,'loss':LOSS,'select_score':sel,'mode':mode}
    accs={}
    for a in ALPHAS:
        acc,n=acc_of(d,a); accs[a]=acc
        rec[f"a={a:+.1f}"]=round(acc,4) if acc==acc else None
    base=accs.get(0.0,float('nan'))
    nz={a:accs[a] for a in ALPHAS if a!=0.0 and accs[a]==accs[a]}
    if nz:
        ba=max(nz,key=nz.get); rec['best_acc']=round(nz[ba],4)
        rec['best_alpha']=f"{ba:+.1f}"; rec['delta_vs0']=round(nz[ba]-base,4) if base==base else None
    rec['n']=acc_of(d,0.0)[1]; rows.append(rec)

df=pd.DataFrame(rows).sort_values(['select_score','mode']).reset_index(drop=True)
out=os.path.join(partb.RUNS_BASE, f"{DS}_{LOSS}_ALL_selectscores_K{K}.xlsx")
df.to_excel(out, index=False)
print(f"{df['select_score'].nunique()} select_score(s) x {df['mode'].nunique()} modes | saved: {out}")
df

4 select_score(s) x 5 modes | saved: $CAGA_BASE/caga_v30_2_runs/mmar_KL_ALL_selectscores_K10000.xlsx


   dataset loss  select_score        mode  a=-1.0  a=-0.3  a=+0.0  a=+0.3  \
0     mmar   KL           aad       audio   0.590   0.631   0.629   0.632   
1     mmar   KL           aad  audio_text   0.530   0.627   0.629   0.639   
2     mmar   KL           aad       final   0.659   0.636   0.629   0.626   
3     mmar   KL           aad   per_token   0.520   0.626   0.629   0.632   
4     mmar   KL           aad        text   0.599   0.635   0.629   0.642   
5     mmar   KL     clean_abs       audio   0.556   0.618   0.629   0.639   
6     mmar   KL     clean_abs  audio_text   0.426   0.614   0.629   0.630   
7     mmar   KL     clean_abs       final   0.652   0.641   0.629   0.626   
8     mmar   KL     clean_abs   per_token   0.529   0.622   0.629   0.634   
9     mmar   KL     clean_abs        text   0.567   0.632   0.629   0.639   
10    mmar   KL  clean_signed       audio   0.335   0.393   0.629   0.599   
11    mmar   KL  clean_signed  audio_text   0.291   0.354   0.629   0.592   

In [ ]:
# Fix one mode, one row per select_score method, compare alphas
import os, json, glob
import pandas as pd
from caga import partb
from caga.adapter import alpha_tag

FIX_MODE = 'audio'          # <- change here: final / audio / text / audio_text / per_token
DS = 'mmar'; K = partb.V30_K_LIST[0]; LOSS = 'KL'

base_dir = os.path.join(partb.RUNS_BASE, "caga_eval_v30_2", DS)
ALPHAS = partb.V30_ALPHAS
KNOWN = ['final','audio','text','audio_text','per_token']

def parse_dir(name):
    core = name[len("gradxact"):].rsplit(f"_none_K{K}", 1)[0].lstrip("_")
    for mode in sorted(KNOWN, key=len, reverse=True):
        if core == mode:            return ('contrast', mode)
        if core.endswith("_"+mode): return (core[:-(len(mode)+1)], mode)
    return (core, '?')

def acc_of(d, a):
    p = os.path.join(d, f"alpha_{alpha_tag(a)}.jsonl")
    if not os.path.exists(p): return (float('nan'), 0)
    n=c=0
    for line in open(p):
        line=line.strip()
        if not line: continue
        r=json.loads(line); n+=1; c+=int(bool(r.get('correct')))
    return (c/n if n else float('nan'), n)

# Collect the results of every select_score under FIX_MODE
rows=[]
for d in sorted(glob.glob(os.path.join(base_dir, f"gradxact*_none_K{K}*"))):
    sel, mode = parse_dir(os.path.basename(d))
    if mode != FIX_MODE: continue
    rec={'select_score':sel}; accs={}
    for a in ALPHAS:
        acc,n = acc_of(d,a); accs[a]=acc
        rec[f"a={a:+.1f}"]=round(acc,4) if acc==acc else None
    base=accs.get(0.0,float('nan'))
    nz={a:accs[a] for a in ALPHAS if a!=0.0 and accs[a]==accs[a]}
    if nz:
        ba=max(nz,key=nz.get); rec['best_acc']=round(nz[ba],4)
        rec['best_alpha']=f"{ba:+.1f}"; rec['delta_vs0']=round(nz[ba]-base,4) if base==base else None
    rec['n']=acc_of(d,0.0)[1]; rows.append(rec)

# Sort: methods in the usual order
order = ['contrast','clean_abs','clean_signed','noise_abs','noise_signed','aad']
df=pd.DataFrame(rows)
df['_o']=df['select_score'].apply(lambda s: order.index(s) if s in order else 99)
df=df.sort_values('_o').drop(columns='_o').reset_index(drop=True)

out=os.path.join(partb.RUNS_BASE, f"{DS}_{LOSS}_mode-{FIX_MODE}_by_selectscore_K{K}.xlsx")
df.to_excel(out, index=False)
print(f"mode={FIX_MODE} | baseline(α=0)={acc_of(glob.glob(os.path.join(base_dir,f'gradxact*_{FIX_MODE}_none_K{K}*') if FIX_MODE!='final' else os.path.join(base_dir,f'gradxact_{FIX_MODE}_none_K{K}'))[0] if False else '',0)[0] if False else 'see table'}")
print("saved:", out)
df

/usr/local/lib/python3.13/dist-packages/huggingface_hub/constants.py:299: FutureWarning: The `HF_HUB_ENABLE_HF_TRANSFER` environment variable is deprecated as 'hf_transfer' is not used anymore. Please use `HF_XET_HIGH_PERFORMANCE` instead to enable high performance transfer with Xet. Visit https://huggingface.co/docs/huggingface_hub/package_reference/environment_variables#hfxethighperformance for more details.
  warnings.warn(


config ready | contrast = gaussian_pure | alphas = [-1.0, -0.3, 0.0, 0.3, 1.0] | RUNS_BASE = $CAGA_BASE/caga_v30_2_runs
forward helpers ready (gaussian 'pure' contrast)
v30 methods ready: compute_scope_attribution (gradxact) / build_specs (modes) / PerItemRunner
mode=audio | baseline(α=0)=see table
saved: $CAGA_BASE/caga_v30_2_runs/mmar_KL_mode-audio_by_selectscore_K10000.xlsx


   select_score  a=-1.0  a=-0.3  a=+0.0  a=+0.3  a=+1.0  best_acc best_alpha  \
0     clean_abs   0.556   0.618   0.629   0.639   0.625     0.639       +0.3   
1  clean_signed   0.335   0.393   0.629   0.599   0.582     0.599       +0.3   
2  noise_signed   0.619   0.629   0.629   0.640   0.630     0.640       +0.3   
3           aad   0.590   0.631   0.629   0.632   0.621     0.632       +0.3   

   delta_vs0     n  
0      0.010  1000  
1     -0.030  1000  
2      0.011  1000  
3      0.003  1000  

In [ ]:
# 5 modes, one sheet each (one row per select_score), all in one Excel file
import os, json, glob
import pandas as pd
from caga import partb
from caga.adapter import alpha_tag

DS = 'mmar'; K = partb.V30_K_LIST[0]; LOSS = 'KL'
MODES = ['final','audio','text','audio_text','per_token']
ORDER = ['contrast','clean_abs','clean_signed','noise_abs','noise_signed','aad']
base_dir = os.path.join(partb.RUNS_BASE, "caga_eval_v30_2", DS)
ALPHAS = partb.V30_ALPHAS

def parse_dir(name):
    core = name[len("gradxact"):].rsplit(f"_none_K{K}", 1)[0].lstrip("_")
    for mode in sorted(MODES, key=len, reverse=True):
        if core == mode:            return ('contrast', mode)
        if core.endswith("_"+mode): return (core[:-(len(mode)+1)], mode)
    return (core, '?')

def acc_of(d, a):
    p = os.path.join(d, f"alpha_{alpha_tag(a)}.jsonl")
    if not os.path.exists(p): return (float('nan'), 0)
    n=c=0
    for line in open(p):
        line=line.strip()
        if line: r=json.loads(line); n+=1; c+=int(bool(r.get('correct')))
    return (c/n if n else float('nan'), n)

# Collect: {mode: {select: {alpha: acc}}}
data = {m: {} for m in MODES}
for d in sorted(glob.glob(os.path.join(base_dir, f"gradxact*_none_K{K}*"))):
    sel, mode = parse_dir(os.path.basename(d))
    if mode not in data: continue
    data[mode][sel] = {a: acc_of(d, a) for a in ALPHAS}

def make_df(mode):
    rows=[]
    for sel in ORDER:
        if sel not in data[mode]: continue
        accs = {a: data[mode][sel][a][0] for a in ALPHAS}
        rec = {'select_score': sel}
        for a in ALPHAS: rec[f"a={a:+.1f}"] = round(accs[a],4) if accs[a]==accs[a] else None
        base = accs.get(0.0, float('nan'))
        nz = {a:accs[a] for a in ALPHAS if a!=0.0 and accs[a]==accs[a]}
        if nz:
            ba=max(nz,key=nz.get); rec['best_acc']=round(nz[ba],4)
            rec['best_alpha']=f"{ba:+.1f}"; rec['delta_vs0']=round(nz[ba]-base,4) if base==base else None
        rec['n']=data[mode][sel][0.0][1]
        rows.append(rec)
    return pd.DataFrame(rows)

out = os.path.join(partb.RUNS_BASE, f"{DS}_{LOSS}_by_mode_selectscore_K{K}.xlsx")
with pd.ExcelWriter(out) as xw:
    # One sheet per mode
    for m in MODES:
        df = make_df(m)
        if len(df): df.to_excel(xw, sheet_name=m[:31], index=False)
    # Plus a summary sheet (all mode x select stacked)
    allrows=[]
    for m in MODES:
        df=make_df(m); df.insert(0,'mode',m); allrows.append(df)
    if allrows:
        pd.concat(allrows, ignore_index=True).to_excel(xw, sheet_name='ALL', index=False)

print("saved:", out)
print("Sheets:", MODES, "+ ALL")
# Also display the audio sheet in the notebook
make_df('audio')

saved: $CAGA_BASE/caga_v30_2_runs/mmar_KL_by_mode_selectscore_K10000.xlsx
Sheets: ['final', 'audio', 'text', 'audio_text', 'per_token'] + ALL


   select_score  a=-1.0  a=-0.3  a=+0.0  a=+0.3  a=+1.0  best_acc best_alpha  \
0     clean_abs   0.556   0.618   0.629   0.639   0.625     0.639       +0.3   
1  clean_signed   0.335   0.393   0.629   0.599   0.582     0.599       +0.3   
2  noise_signed   0.619   0.629   0.629   0.640   0.630     0.640       +0.3   
3           aad   0.590   0.631   0.629   0.632   0.621     0.632       +0.3   

   delta_vs0     n  
0      0.010  1000  
1     -0.030  1000  
2      0.011  1000  
3      0.003  1000  

In [ ]:
import re, inspect, caga.data as cd

src = inspect.getsource(cd)

# 1) Print build() itself to see how it dispatches
print(inspect.getsource(cd.build)[:3000])

# 2) Guess automatically: build_xxx functions + dicts/lists containing 'mmar'
cands = set(re.findall(r"def _?(?:build|prep|prepare|load)_(\w+)", src))
for n in dir(cd):
    o = getattr(cd, n)
    if isinstance(o, (dict, list, tuple, set)) and 'mmar' in o:
        print(f"\n[{n}] ->", list(o.keys()) if isinstance(o, dict) else list(o))
        cands |= set(map(str, o.keys() if isinstance(o, dict) else o))
print("\nPossible datasets:", sorted(cands))

# 3) What is already built under CAGA_BASE
import os
d = os.path.join(os.environ["CAGA_BASE"], "datasets")
print("Already built:", sorted(os.listdir(d)) if os.path.isdir(d) else "none")

def build(*names):
    """build('mmau_mini','mmar', ...) or build() for all benchmarks."""
    names = names or ALL
    for n in names:
        print(f"\n########## building {n} ##########")
        REGISTRY[n].build()


[ALL] -> ['mmau_mini', 'mmar', 'mmsu', 'ravdess', 'iemocap', 'meld', 'cremad']

[REGISTRY] -> ['mmau_mini', 'mmar', 'mmsu', 'ravdess', 'iemocap', 'meld', 'airbench_foundation', 'cremad']

Possible datasets: ['airbench_foundation', 'cremad', 'iemocap', 'meld', 'mmar', 'mmau_mini', 'mmsu', 'ravdess']
Already built: ['mmar']


other dataset

In [ ]:
# ===== One cell: choose dataset -> build -> steering -> eval -> save Excel =====
DS     = 'mmsu'                                   # <- name of the dataset to run
TO_RUN = ['contrast', 'clean_abs', 'clean_signed',
          'noise_abs', 'noise_signed', 'aad']     # <- which SELECT_SCOREs to run
LOSS   = 'KL'                                     # only a label written into the table

import os, re, json, glob, inspect
import pandas as pd
import caga.data as cd
from caga.data import build, data_summary
from caga import partb
from caga.adapter import alpha_tag

# ---- 1. Build the dataset (download is skipped if the manifest already exists) ----
try:
    build(DS)
except Exception as e:
    src = inspect.getsource(cd)
    cands = set(re.findall(r"def _?build_(\w+)", src))
    for n in dir(cd):
        o = getattr(cd, n)
        if isinstance(o, dict) and 'mmar' in o: cands |= set(map(str, o.keys()))
    print(f"build('{DS}') failed: {e}\npossibly supported names: {sorted(cands)}")
    raise
data_summary([DS])
man = os.path.join(os.environ["CAGA_BASE"], "datasets", DS, "manifest.jsonl")
assert os.path.exists(man), f"not found: {man}"

# ---- 2. Load the model and run each SELECT_SCORE in turn (resumable) ----
partb.load_model()                      # returns immediately if already loaded
for sel in TO_RUN:
    partb.SELECT_SCORE = sel
    print(f"\n########## {DS} | SELECT_SCORE = {sel} ##########")
    partb.run_v30_2([DS])
    partb.eval_v30_2([DS])

# ---- 3. Write Excel (one sheet per mode + ALL) ----
K = partb.V30_K_LIST[0]
ALPHAS = partb.V30_ALPHAS
MODES = ['final', 'audio', 'text', 'audio_text', 'per_token']
ORDER = ['contrast', 'clean_abs', 'clean_signed', 'noise_abs', 'noise_signed', 'aad']
base_dir = os.path.join(partb.RUNS_BASE, "caga_eval_v30_2", DS)

def parse_dir(name):
    core = name[len("gradxact"):].rsplit(f"_none_K{K}", 1)[0].lstrip("_")
    for m in sorted(MODES, key=len, reverse=True):
        if core == m:              return ('contrast', m)
        if core.endswith("_" + m): return (core[:-(len(m) + 1)], m)
    return (core, '?')

def acc_of(d, a):
    p = os.path.join(d, f"alpha_{alpha_tag(a)}.jsonl")
    if not os.path.exists(p): return (float('nan'), 0)
    n = c = 0
    for line in open(p):
        line = line.strip()
        if line:
            r = json.loads(line); n += 1; c += int(bool(r.get('correct')))
    return (c / n if n else float('nan'), n)

data = {m: {} for m in MODES}
for d in sorted(glob.glob(os.path.join(base_dir, f"gradxact*_none_K{K}*"))):
    sel, m = parse_dir(os.path.basename(d))
    if m in data: data[m][sel] = {a: acc_of(d, a) for a in ALPHAS}

def make_df(m):
    rows = []
    for sel in sorted(data[m], key=lambda s: ORDER.index(s) if s in ORDER else 99):
        accs = {a: data[m][sel][a][0] for a in ALPHAS}
        rec = {'dataset': DS, 'loss': LOSS, 'mode': m, 'select_score': sel}
        for a in ALPHAS:
            rec[f"a={a:+.1f}"] = round(accs[a], 4) if accs[a] == accs[a] else None
        base = accs.get(0.0, float('nan'))
        nz = {a: v for a, v in accs.items() if a != 0.0 and v == v}
        if nz:
            ba = max(nz, key=nz.get)
            rec['best_acc'] = round(nz[ba], 4)
            rec['best_alpha'] = f"{ba:+.1f}"
            rec['delta_vs0'] = round(nz[ba] - base, 4) if base == base else None
        rec['n'] = data[m][sel].get(0.0, (0, 0))[1]
        rows.append(rec)
    return pd.DataFrame(rows)

out = os.path.join(partb.RUNS_BASE, f"{DS}_{LOSS}_by_mode_selectscore_K{K}.xlsx")
dfs = [make_df(m) for m in MODES]
with pd.ExcelWriter(out) as xw:
    for m, df in zip(MODES, dfs):
        if len(df): df.to_excel(xw, sheet_name=m[:31], index=False)
    pd.concat([d for d in dfs if len(d)], ignore_index=True).to_excel(xw, sheet_name='ALL', index=False)
print("\nsaved:", out)
pd.concat([d for d in dfs if len(d)], ignore_index=True)

/usr/local/lib/python3.13/dist-packages/huggingface_hub/constants.py:299: FutureWarning: The `HF_HUB_ENABLE_HF_TRANSFER` environment variable is deprecated as 'hf_transfer' is not used anymore. Please use `HF_XET_HIGH_PERFORMANCE` instead to enable high performance transfer with Xet. Visit https://huggingface.co/docs/huggingface_hub/package_reference/environment_variables#hfxethighperformance for more details.
  warnings.warn(


config ready | contrast = gaussian_pure | alphas = [-1.0, -0.3, 0.0, 0.3, 1.0] | RUNS_BASE = $CAGA_BASE/caga_v30_2_runs
forward helpers ready (gaussian 'pure' contrast)
v30 methods ready: compute_scope_attribution (gradxact) / build_specs (modes) / PerItemRunner

########## building mmsu ##########


README.md:   0%|          | 0.00/2.39k [00:00<?, ?B/s]

data/train-00000-of-00003.parquet: reconstructing file:   0%|          |  0.00B /  425MB            

data/train-00000-of-00003.parquet: downloading bytes:           |  0.00B            

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/1346 [00:00<?, ?it/s]

[Load] done

########## mmsu | SELECT_SCORE = contrast ##########

=== mmsu: 5000 items | modes=['final', 'audio', 'text', 'audio_text', 'per_token'] | K=[10000] | 5 configs x 5 alphas | 2 fwd-bwd/item (gradxact) ===
  [sanity] id=mmsu/00000 grad-L2 audio/pos=0.02921 text/pos=0.18122 final=0.422 T=143
    [mmsu] 10 items (46s)
    [mmsu] 20 items (74s)
    [mmsu] 30 items (104s)
    [mmsu] 40 items (134s)
    [mmsu] 50 items (162s)
    [mmsu] 60 items (190s)
    [mmsu] 70 items (218s)
    [mmsu] 80 items (247s)
    [mmsu] 90 items (274s)
    [mmsu] 100 items (304s)
    [mmsu] 110 items (332s)
    [mmsu] 120 items (364s)
    [mmsu] 130 items (394s)
    [mmsu] 140 items (427s)
    [mmsu] 150 items (457s)
    [mmsu] 160 items (488s)
    [mmsu] 170 items (516s)
    [mmsu] 180 items (547s)
    [mmsu] 190 items (579s)
    [mmsu] 200 items (607s)
    [mmsu] 210 items (637s)
    [mmsu] 220 items (669s)
    [mmsu] 230 items (707s)
    [mmsu] 240 items (747s)
    [mmsu] 250 items (789s)
    [mms

KeyboardInterrupt: 

In [ ]:
# ===== One cell: choose dataset -> build -> steering -> eval -> save Excel =====
DS     = 'iemocap'                                   # <- name of the dataset to run
TO_RUN = ['clean_abs', 'clean_signed',
          'noise_abs', 'aad']     # <- which SELECT_SCOREs to run
LOSS   = 'KL'                                     # only a label written into the table

import os, re, json, glob, inspect
import pandas as pd
import caga.data as cd
from caga.data import build, data_summary
from caga import partb
from caga.adapter import alpha_tag

# ---- 1. Build the dataset (download is skipped if the manifest already exists) ----
try:
    build(DS)
except Exception as e:
    src = inspect.getsource(cd)
    cands = set(re.findall(r"def _?build_(\w+)", src))
    for n in dir(cd):
        o = getattr(cd, n)
        if isinstance(o, dict) and 'mmar' in o: cands |= set(map(str, o.keys()))
    print(f"build('{DS}') failed: {e}\npossibly supported names: {sorted(cands)}")
    raise
data_summary([DS])
man = os.path.join(os.environ["CAGA_BASE"], "datasets", DS, "manifest.jsonl")
assert os.path.exists(man), f"not found: {man}"

# ---- 2. Load the model and run each SELECT_SCORE in turn (resumable) ----
partb.load_model()                      # returns immediately if already loaded
for sel in TO_RUN:
    partb.SELECT_SCORE = sel
    print(f"\n########## {DS} | SELECT_SCORE = {sel} ##########")
    partb.run_v30_2([DS])
    partb.eval_v30_2([DS])

# ---- 3. Write Excel (one sheet per mode + ALL) ----
K = partb.V30_K_LIST[0]
ALPHAS = partb.V30_ALPHAS
MODES = ['final', 'audio', 'text', 'audio_text', 'per_token']
ORDER = ['contrast', 'clean_abs', 'clean_signed', 'noise_abs', 'noise_signed', 'aad']
base_dir = os.path.join(partb.RUNS_BASE, "caga_eval_v30_2", DS)

def parse_dir(name):
    core = name[len("gradxact"):].rsplit(f"_none_K{K}", 1)[0].lstrip("_")
    for m in sorted(MODES, key=len, reverse=True):
        if core == m:              return ('contrast', m)
        if core.endswith("_" + m): return (core[:-(len(m) + 1)], m)
    return (core, '?')

def acc_of(d, a):
    p = os.path.join(d, f"alpha_{alpha_tag(a)}.jsonl")
    if not os.path.exists(p): return (float('nan'), 0)
    n = c = 0
    for line in open(p):
        line = line.strip()
        if line:
            r = json.loads(line); n += 1; c += int(bool(r.get('correct')))
    return (c / n if n else float('nan'), n)

data = {m: {} for m in MODES}
for d in sorted(glob.glob(os.path.join(base_dir, f"gradxact*_none_K{K}*"))):
    sel, m = parse_dir(os.path.basename(d))
    if m in data: data[m][sel] = {a: acc_of(d, a) for a in ALPHAS}

def make_df(m):
    rows = []
    for sel in sorted(data[m], key=lambda s: ORDER.index(s) if s in ORDER else 99):
        accs = {a: data[m][sel][a][0] for a in ALPHAS}
        rec = {'dataset': DS, 'loss': LOSS, 'mode': m, 'select_score': sel}
        for a in ALPHAS:
            rec[f"a={a:+.1f}"] = round(accs[a], 4) if accs[a] == accs[a] else None
        base = accs.get(0.0, float('nan'))
        nz = {a: v for a, v in accs.items() if a != 0.0 and v == v}
        if nz:
            ba = max(nz, key=nz.get)
            rec['best_acc'] = round(nz[ba], 4)
            rec['best_alpha'] = f"{ba:+.1f}"
            rec['delta_vs0'] = round(nz[ba] - base, 4) if base == base else None
        rec['n'] = data[m][sel].get(0.0, (0, 0))[1]
        rows.append(rec)
    return pd.DataFrame(rows)

out = os.path.join(partb.RUNS_BASE, f"{DS}_{LOSS}_by_mode_selectscore_K{K}.xlsx")
dfs = [make_df(m) for m in MODES]
with pd.ExcelWriter(out) as xw:
    for m, df in zip(MODES, dfs):
        if len(df): df.to_excel(xw, sheet_name=m[:31], index=False)
    pd.concat([d for d in dfs if len(d)], ignore_index=True).to_excel(xw, sheet_name='ALL', index=False)
print("\nsaved:", out)
pd.concat([d for d in dfs if len(d)], ignore_index=True)


########## building iemocap ##########


README.md:   0%|          | 0.00/1.07k [00:00<?, ?B/s]

data/train-00000-of-00003.parquet: reconstructing file:   0%|          |  0.00B /  489MB            

data/train-00000-of-00003.parquet: downloading bytes:           |  0.00B            

data/train-00001-of-00003.parquet: reconstructing file:   0%|          |  0.00B /  456MB            

...

data/train-00002-of-00003.parquet: reconstructing file:   0%|          |  0.00B /  462MB            

data/train-00002-of-00003.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/10039 [00:00<?, ? examples/s]

[iemocap] wrote 5528 rows -> $CAGA_BASE/datasets/iemocap/manifest.jsonl
[iemocap] 5528 items | missing audio: 0
   e.g. {'id': 'iemocap/Ses01F_impro01_F000', 'domain': 'speech', 'choices': ['angry', 'happy', 'sad', 'neutral'], 'gold_idx': 3}
[iemocap] 5528 (standard 4-class; canonical=5531)
dataset                  items  audio_ok   status
------------------------------------------------------
iemocap                   5528      5528   OK

schema: id · dataset · audio_path · question · choices · gold_idx · domain

########## iemocap | SELECT_SCORE = clean_abs ##########

=== iemocap: 5528 items | modes=['final', 'audio', 'text', 'audio_text', 'per_token'] | K=[10000] | 5 configs x 5 alphas | 2 fwd-bwd/item (gradxact) ===
  [sanity] id=iemocap/Ses01F_impro01_F000 grad-L2 audio/pos=0.02166 text/pos=0.06917 final=0.275 T=102
    [iemocap] 10 items (26s)
    [iemocap] 20 items (53s)
    [iemocap] 30 items (79s)
    [iemocap] 40 items (104s)
    [iemocap] 50 items (128s)
    [iemocap] 60 it

    dataset loss        mode  select_score  a=-1.0  a=-0.3  a=+0.0  a=+0.3  \
0   iemocap   KL       final     clean_abs  0.7071  0.7028  0.6927  0.6843   
1   iemocap   KL       final  clean_signed  0.6666  0.6977  0.6927  0.6832   
2   iemocap   KL       final     noise_abs  0.7080  0.7001  0.6927  0.6856   
3   iemocap   KL       final           aad  0.7100  0.7003  0.6927  0.6861   
4   iemocap   KL       audio     clean_abs  0.5304  0.6764  0.6927  0.6954   
5   iemocap   KL       audio  clean_signed  0.2314  0.2851  0.6927  0.6286   
6   iemocap   KL       audio     noise_abs  0.6532  0.6903  0.6927  0.6959   
7   iemocap   KL       audio           aad  0.5966  0.6793  0.6927  0.6977   
8   iemocap   KL        text     clean_abs  0.6185  0.7001  0.6927  0.6823   
9   iemocap   KL        text  clean_signed  0.3603  0.6346  0.6927  0.6552   
10  iemocap   KL        text     noise_abs  0.6440  0.7013  0.6927  0.6823   
11  iemocap   KL        text           aad  0.6974  0.6963  0.69

In [ ]:
# ===== One cell: choose dataset -> build -> steering -> eval -> save Excel =====
DS     = 'meld'                                   # <- name of the dataset to run
TO_RUN = ['contrast', 'clean_abs', 'clean_signed',
          'noise_abs', 'noise_signed', 'aad']     # <- which SELECT_SCOREs to run
LOSS   = 'KL'                                     # only a label written into the table

import os, re, json, glob, inspect
import pandas as pd
import caga.data as cd
from caga.data import build, data_summary
from caga import partb
from caga.adapter import alpha_tag

# ---- 1. Build the dataset (download is skipped if the manifest already exists) ----
try:
    build(DS)
except Exception as e:
    src = inspect.getsource(cd)
    cands = set(re.findall(r"def _?build_(\w+)", src))
    for n in dir(cd):
        o = getattr(cd, n)
        if isinstance(o, dict) and 'mmar' in o: cands |= set(map(str, o.keys()))
    print(f"build('{DS}') failed: {e}\npossibly supported names: {sorted(cands)}")
    raise
data_summary([DS])
man = os.path.join(os.environ["CAGA_BASE"], "datasets", DS, "manifest.jsonl")
assert os.path.exists(man), f"not found: {man}"

# ---- 2. Load the model and run each SELECT_SCORE in turn (resumable) ----
partb.load_model()                      # returns immediately if already loaded
for sel in TO_RUN:
    partb.SELECT_SCORE = sel
    print(f"\n########## {DS} | SELECT_SCORE = {sel} ##########")
    partb.run_v30_2([DS])
    partb.eval_v30_2([DS])

# ---- 3. Write Excel (one sheet per mode + ALL) ----
K = partb.V30_K_LIST[0]
ALPHAS = partb.V30_ALPHAS
MODES = ['final', 'audio', 'text', 'audio_text', 'per_token']
ORDER = ['contrast', 'clean_abs', 'clean_signed', 'noise_abs', 'noise_signed', 'aad']
base_dir = os.path.join(partb.RUNS_BASE, "caga_eval_v30_2", DS)

def parse_dir(name):
    core = name[len("gradxact"):].rsplit(f"_none_K{K}", 1)[0].lstrip("_")
    for m in sorted(MODES, key=len, reverse=True):
        if core == m:              return ('contrast', m)
        if core.endswith("_" + m): return (core[:-(len(m) + 1)], m)
    return (core, '?')

def acc_of(d, a):
    p = os.path.join(d, f"alpha_{alpha_tag(a)}.jsonl")
    if not os.path.exists(p): return (float('nan'), 0)
    n = c = 0
    for line in open(p):
        line = line.strip()
        if line:
            r = json.loads(line); n += 1; c += int(bool(r.get('correct')))
    return (c / n if n else float('nan'), n)

data = {m: {} for m in MODES}
for d in sorted(glob.glob(os.path.join(base_dir, f"gradxact*_none_K{K}*"))):
    sel, m = parse_dir(os.path.basename(d))
    if m in data: data[m][sel] = {a: acc_of(d, a) for a in ALPHAS}

def make_df(m):
    rows = []
    for sel in sorted(data[m], key=lambda s: ORDER.index(s) if s in ORDER else 99):
        accs = {a: data[m][sel][a][0] for a in ALPHAS}
        rec = {'dataset': DS, 'loss': LOSS, 'mode': m, 'select_score': sel}
        for a in ALPHAS:
            rec[f"a={a:+.1f}"] = round(accs[a], 4) if accs[a] == accs[a] else None
        base = accs.get(0.0, float('nan'))
        nz = {a: v for a, v in accs.items() if a != 0.0 and v == v}
        if nz:
            ba = max(nz, key=nz.get)
            rec['best_acc'] = round(nz[ba], 4)
            rec['best_alpha'] = f"{ba:+.1f}"
            rec['delta_vs0'] = round(nz[ba] - base, 4) if base == base else None
        rec['n'] = data[m][sel].get(0.0, (0, 0))[1]
        rows.append(rec)
    return pd.DataFrame(rows)

out = os.path.join(partb.RUNS_BASE, f"{DS}_{LOSS}_by_mode_selectscore_K{K}.xlsx")
dfs = [make_df(m) for m in MODES]
with pd.ExcelWriter(out) as xw:
    for m, df in zip(MODES, dfs):
        if len(df): df.to_excel(xw, sheet_name=m[:31], index=False)
    pd.concat([d for d in dfs if len(d)], ignore_index=True).to_excel(xw, sheet_name='ALL', index=False)
print("\nsaved:", out)
pd.concat([d for d in dfs if len(d)], ignore_index=True)

In [ ]:
# ===== One cell: choose dataset -> build -> steering -> eval -> save Excel =====
DS     = 'mmau_mini'                                   # <- name of the dataset to run
TO_RUN = ['contrast', 'clean_abs', 'clean_signed',
          'noise_abs', 'noise_signed', 'aad']     # <- which SELECT_SCOREs to run
LOSS   = 'KL'                                     # only a label written into the table

import os, re, json, glob, inspect
import pandas as pd
import caga.data as cd
from caga.data import build, data_summary
from caga import partb
from caga.adapter import alpha_tag

# ---- 1. Build the dataset (download is skipped if the manifest already exists) ----
try:
    build(DS)
except Exception as e:
    src = inspect.getsource(cd)
    cands = set(re.findall(r"def _?build_(\w+)", src))
    for n in dir(cd):
        o = getattr(cd, n)
        if isinstance(o, dict) and 'mmar' in o: cands |= set(map(str, o.keys()))
    print(f"build('{DS}') failed: {e}\npossibly supported names: {sorted(cands)}")
    raise
data_summary([DS])
man = os.path.join(os.environ["CAGA_BASE"], "datasets", DS, "manifest.jsonl")
assert os.path.exists(man), f"not found: {man}"

# ---- 2. Load the model and run each SELECT_SCORE in turn (resumable) ----
partb.load_model()                      # returns immediately if already loaded
for sel in TO_RUN:
    partb.SELECT_SCORE = sel
    print(f"\n########## {DS} | SELECT_SCORE = {sel} ##########")
    partb.run_v30_2([DS])
    partb.eval_v30_2([DS])

# ---- 3. Write Excel (one sheet per mode + ALL) ----
K = partb.V30_K_LIST[0]
ALPHAS = partb.V30_ALPHAS
MODES = ['final', 'audio', 'text', 'audio_text', 'per_token']
ORDER = ['contrast', 'clean_abs', 'clean_signed', 'noise_abs', 'noise_signed', 'aad']
base_dir = os.path.join(partb.RUNS_BASE, "caga_eval_v30_2", DS)

def parse_dir(name):
    core = name[len("gradxact"):].rsplit(f"_none_K{K}", 1)[0].lstrip("_")
    for m in sorted(MODES, key=len, reverse=True):
        if core == m:              return ('contrast', m)
        if core.endswith("_" + m): return (core[:-(len(m) + 1)], m)
    return (core, '?')

def acc_of(d, a):
    p = os.path.join(d, f"alpha_{alpha_tag(a)}.jsonl")
    if not os.path.exists(p): return (float('nan'), 0)
    n = c = 0
    for line in open(p):
        line = line.strip()
        if line:
            r = json.loads(line); n += 1; c += int(bool(r.get('correct')))
    return (c / n if n else float('nan'), n)

data = {m: {} for m in MODES}
for d in sorted(glob.glob(os.path.join(base_dir, f"gradxact*_none_K{K}*"))):
    sel, m = parse_dir(os.path.basename(d))
    if m in data: data[m][sel] = {a: acc_of(d, a) for a in ALPHAS}

def make_df(m):
    rows = []
    for sel in sorted(data[m], key=lambda s: ORDER.index(s) if s in ORDER else 99):
        accs = {a: data[m][sel][a][0] for a in ALPHAS}
        rec = {'dataset': DS, 'loss': LOSS, 'mode': m, 'select_score': sel}
        for a in ALPHAS:
            rec[f"a={a:+.1f}"] = round(accs[a], 4) if accs[a] == accs[a] else None
        base = accs.get(0.0, float('nan'))
        nz = {a: v for a, v in accs.items() if a != 0.0 and v == v}
        if nz:
            ba = max(nz, key=nz.get)
            rec['best_acc'] = round(nz[ba], 4)
            rec['best_alpha'] = f"{ba:+.1f}"
            rec['delta_vs0'] = round(nz[ba] - base, 4) if base == base else None
        rec['n'] = data[m][sel].get(0.0, (0, 0))[1]
        rows.append(rec)
    return pd.DataFrame(rows)

out = os.path.join(partb.RUNS_BASE, f"{DS}_{LOSS}_by_mode_selectscore_K{K}.xlsx")
dfs = [make_df(m) for m in MODES]
with pd.ExcelWriter(out) as xw:
    for m, df in zip(MODES, dfs):
        if len(df): df.to_excel(xw, sheet_name=m[:31], index=False)
    pd.concat([d for d in dfs if len(d)], ignore_index=True).to_excel(xw, sheet_name='ALL', index=False)
print("\nsaved:", out)
pd.concat([d for d in dfs if len(d)], ignore_index=True)